# Análise do patrimônio dos candidatos do Amazonas — Eleições 2026

Este notebook realiza a análise dos dados de candidatos e bens declarados nas Eleições 2026 para o estado do Amazonas.

Serão utilizados dois arquivos disponibilizados pelo Tribunal Superior Eleitoral (TSE) (https://dadosabertos.tse.jus.br/dataset/candidatos-2026): a base de candidatos e a base de bens. As bases serão relacionadas por meio da chave `SQ_CANDIDATO`.

O objetivo é analisar a distribuição dos bens declarados e investigar diferenças de patrimônio entre grupos de candidatos, como cargo, partido e gênero.

## A1. Carga e inspeção inicial

Nesta etapa serão carregados os dois arquivos disponibilizados pelo TSE.

A base de candidatos contém informações como partido, cargo, gênero e nome de urna. A base de bens contém os bens declarados por cada candidato e seus respectivos valores.

Os arquivos serão lidos utilizando `;` como separador e `latin-1` como codificação. No arquivo de bens também será utilizado `decimal=","`, pois os valores monetários utilizam vírgula como separador decimal.

A coluna `SQ_CANDIDATO` será lida como texto nos dois arquivos para garantir que possa ser utilizada corretamente como chave do JOIN.

In [ ]:
# Import de bibliotecas
import pandas as pd
import plotly.express as px

In [2]:
#  Carregar dados dos candidatos
candidatos_df = pd.read_csv('consulta_cand_2026_AM.csv', sep=';', encoding='latin-1', dtype={'SQ_CANDIDATO': str})

#  Carregar dados dos candidatos
bens_df = pd.read_csv('bem_candidato_2026_AM.csv', sep=';', encoding='latin-1', decimal=',', dtype={'SQ_CANDIDATO': str})

### Inspeção das bases

Após a leitura, serão verificadas as dimensões (`shape`), algumas linhas iniciais (`head()`) e os tipos de dados (`dtypes`) de cada DataFrame.

Essa inspeção permite verificar se os arquivos foram carregados corretamente e se as colunas relevantes possuem os tipos esperados.

In [3]:
# Verificar dados de candidatos
print("===== CANDIDATOS =====\n")
print("Shape:", candidatos_df.shape)
print("\nDtypes:")
print(candidatos_df.dtypes)
display(candidatos_df.head())

===== CANDIDATOS =====

Shape: (469, 50)

Dtypes:
DT_GERACAO                         str
HH_GERACAO                         str
ANO_ELEICAO                      int64
CD_TIPO_ELEICAO                  int64
NM_TIPO_ELEICAO                    str
NR_TURNO                         int64
CD_ELEICAO                       int64
DS_ELEICAO                         str
DT_ELEICAO                         str
TP_ABRANGENCIA                     str
SG_UF                              str
SG_UE                              str
NM_UE                              str
CD_CARGO                         int64
DS_CARGO                           str
SQ_CANDIDATO                       str
NR_CANDIDATO                     int64
NM_CANDIDATO                       str
NM_URNA_CANDIDATO                  str
NM_SOCIAL_CANDIDATO                str
NR_CPF_CANDIDATO                 int64
DS_EMAIL                           str
CD_SITUACAO_CANDIDATURA          int64
DS_SITUACAO_CANDIDATURA            str
TP_AGREMIACAO 

,DT_GERACAO,HH_GERACAO,ANO_ELEICAO,CD_TIPO_ELEICAO,NM_TIPO_ELEICAO,NR_TURNO,CD_ELEICAO,DS_ELEICAO,DT_ELEICAO,TP_ABRANGENCIA,...,CD_GRAU_INSTRUCAO,DS_GRAU_INSTRUCAO,CD_ESTADO_CIVIL,DS_ESTADO_CIVIL,CD_COR_RACA,DS_COR_RACA,CD_OCUPACAO,DS_OCUPACAO,CD_SIT_TOT_TURNO,DS_SIT_TOT_TURNO
0,31/08/2026,12:30:43,2026,2,ELEIÇÃO ORDINÁRIA,1,6259,Eleições Gerais Estaduais 2026,04/10/2026,ESTADUAL,...,8,SUPERIOR COMPLETO,3,CASADO(A),5,INDÍGENA,999,OUTROS,-1,#NULO
1,31/08/2026,12:30:43,2026,2,ELEIÇÃO ORDINÁRIA,1,6259,Eleições Gerais Estaduais 2026,04/10/2026,ESTADUAL,...,6,ENSINO MÉDIO COMPLETO,1,SOLTEIRO(A),3,PARDA,155,TÉCNICO DE MECÂNICA,-1,#NULO
2,31/08/2026,12:30:43,2026,2,ELEIÇÃO ORDINÁRIA,1,6259,Eleições Gerais Estaduais 2026,04/10/2026,ESTADUAL,...,8,SUPERIOR COMPLETO,1,SOLTEIRO(A),5,INDÍGENA,297,SERVIDOR PÚBLICO ESTADUAL,-1,#NULO
3,31/08/2026,12:30:43,2026,2,ELEIÇÃO ORDINÁRIA,1,6259,Eleições Gerais Estaduais 2026,04/10/2026,ESTADUAL,...,4,ENSINO FUNDAMENTAL COMPLETO,1,SOLTEIRO(A),2,PRETA,601,AGRICULTOR,-1,#NULO
4,31/08/2026,12:30:43,2026,2,ELEIÇÃO ORDINÁRIA,1,6259,Eleições Gerais Estaduais 2026,04/10/2026,ESTADUAL,...,5,ENSINO MÉDIO INCOMPLETO,1,SOLTEIRO(A),1,BRANCA,923,APOSENTADO (EXCETO SERVIDOR PÚBLICO),-1,#NULO


In [4]:
# Verificar dados de bens
print("===== BENS =====\n")
print("Shape:", bens_df.shape)
print("\nDtypes:")
print(bens_df.dtypes)
display(bens_df.head())

===== BENS =====

Shape: (1654, 19)

Dtypes:
DT_GERACAO                        str
HH_GERACAO                        str
ANO_ELEICAO                     int64
CD_TIPO_ELEICAO                 int64
NM_TIPO_ELEICAO                   str
CD_ELEICAO                      int64
DS_ELEICAO                        str
DT_ELEICAO                        str
SG_UF                             str
SG_UE                             str
NM_UE                             str
SQ_CANDIDATO                      str
NR_ORDEM_BEM_CANDIDATO          int64
CD_TIPO_BEM_CANDIDATO           int64
DS_TIPO_BEM_CANDIDATO             str
DS_BEM_CANDIDATO                  str
VR_BEM_CANDIDATO              float64
DT_ULT_ATUAL_BEM_CANDIDATO        str
HH_ULT_ATUAL_BEM_CANDIDATO        str
dtype: object


,DT_GERACAO,HH_GERACAO,ANO_ELEICAO,CD_TIPO_ELEICAO,NM_TIPO_ELEICAO,CD_ELEICAO,DS_ELEICAO,DT_ELEICAO,SG_UF,SG_UE,NM_UE,SQ_CANDIDATO,NR_ORDEM_BEM_CANDIDATO,CD_TIPO_BEM_CANDIDATO,DS_TIPO_BEM_CANDIDATO,DS_BEM_CANDIDATO,VR_BEM_CANDIDATO,DT_ULT_ATUAL_BEM_CANDIDATO,HH_ULT_ATUAL_BEM_CANDIDATO
0,31/08/2026,12:31:08,2026,2,Eleição Ordinária,6259,Eleições Gerais Estaduais 2026,04/10/2026,AM,AM,AMAZONAS,40002531438,2,13,Terreno,TERRENO MEDINDO 10X20 ZONA URBANA,80000.00,24/08/2026,15:21:14
1,31/08/2026,12:31:08,2026,2,Eleição Ordinária,6259,Eleições Gerais Estaduais 2026,04/10/2026,AM,AM,AMAZONAS,40002531438,1,21,"Veículo automotor terrestre: caminhão, automóv...",VEICULO MONTANA PREMIER 2025,140000.00,24/08/2026,15:21:14
2,31/08/2026,12:31:08,2026,2,Eleição Ordinária,6259,Eleições Gerais Estaduais 2026,04/10/2026,AM,AM,AMAZONAS,40002531438,3,12,Casa,CASA,90000.00,24/08/2026,15:21:14
3,31/08/2026,12:31:08,2026,2,Eleição Ordinária,6259,Eleições Gerais Estaduais 2026,04/10/2026,AM,AM,AMAZONAS,40002531439,2,61,Depósito bancário em conta corrente no País,DEPOSITO EM CONTA CORRENTE,1339.16,18/08/2026,21:22:29
4,31/08/2026,12:31:08,2026,2,Eleição Ordinária,6259,Eleições Gerais Estaduais 2026,04/10/2026,AM,AM,AMAZONAS,40002531439,3,61,Depósito bancário em conta corrente no País,DEPOSITO EM CONTA CORRENTE,375.91,18/08/2026,21:22:29


### Verificação do valor dos bens

A coluna `VR_BEM_CANDIDATO` representa o valor declarado de cada bem. Como ela será utilizada posteriormente para cálculos de patrimônio, é necessário confirmar que foi carregada como uma coluna numérica.

In [5]:
print("Tipo da coluna 'VR_BEM_CANDIDATO':", bens_df['VR_BEM_CANDIDATO'].dtype)

print(
    "A coluna é numérica?",
    pd.api.types.is_numeric_dtype(
        bens_df['VR_BEM_CANDIDATO']
    )
)

Tipo da coluna 'VR_BEM_CANDIDATO': float64
A coluna é numérica? True


## A2. Limpeza, qualidade e deduplicação

Nesta etapa serão avaliados possíveis problemas de qualidade dos dados.

Primeiramente serão verificados valores ausentes nas principais colunas utilizadas na análise. Em seguida, será confirmado se os registros pertencem ao estado do Amazonas.

Também será verificada a existência de candidatos repetidos na base de candidatos. Como `SQ_CANDIDATO` identifica unicamente cada candidato, será mantida apenas uma linha por candidato antes do JOIN. Essa deduplicação é necessária para evitar que um mesmo candidato apareça várias vezes no resultado final e faça com que seu patrimônio seja contabilizado ou replicado indevidamente.

In [6]:
# Selecionando colunas relevantes
colunas_bens = [
    'SQ_CANDIDATO',
    'DS_TIPO_BEM_CANDIDATO',
    'VR_BEM_CANDIDATO',
    'SG_UF'
]

colunas_candidatos = [
    'SQ_CANDIDATO',
    'DS_CARGO',
    'SG_PARTIDO',
    'DS_GENERO',
    'DS_GRAU_INSTRUCAO',
    'NM_URNA_CANDIDATO'
]

# Verifica valores ausentes
print("Valores ausentes - BENS:")
print(bens_df[colunas_bens].isna().sum())

print("\nValores ausentes - CANDIDATOS:")
print(candidatos_df[colunas_candidatos].isna().sum())

Valores ausentes - BENS:
SQ_CANDIDATO             0
DS_TIPO_BEM_CANDIDATO    0
VR_BEM_CANDIDATO         0
SG_UF                    0
dtype: int64

Valores ausentes - CANDIDATOS:
SQ_CANDIDATO         0
DS_CARGO             0
SG_PARTIDO           0
DS_GENERO            0
DS_GRAU_INSTRUCAO    0
NM_URNA_CANDIDATO    0
dtype: int64


In [7]:
# Verificando se os bens são do AM
bens_df['SG_UF'].value_counts()

SG_UF
AM    1654
Name: count, dtype: int64

### Deduplicação da base de candidatos

A base de candidatos pode conter mais de um registro para o mesmo `SQ_CANDIDATO`. Como a análise final precisa possuir uma única linha por candidato, os registros serão deduplicados utilizando `SQ_CANDIDATO` como chave.

Essa etapa é realizada antes do JOIN para evitar a multiplicação de registros quando o patrimônio agregado for associado aos candidatos.

In [8]:
# CANDIDATOS
print("Total de linhas no arquivo de candidatos:", len(candidatos_df))
print("Candidatos únicos:", candidatos_df['SQ_CANDIDATO'].nunique())
print("Duplicados:", candidatos_df['SQ_CANDIDATO'].duplicated().sum())

# Removendo duplicados
candidatos_df = candidatos_df.drop_duplicates(
    subset='SQ_CANDIDATO'
)

print("\nLinhas depois:", len(candidatos_df))
print("Candidatos únicos:", candidatos_df['SQ_CANDIDATO'].nunique())

Total de linhas no arquivo de candidatos: 469
Candidatos únicos: 469
Duplicados: 0

Linhas depois: 469
Candidatos únicos: 469


## A3. EDA univariada

Nesta etapa será realizada uma análise exploratória individual das variáveis da base de bens.

Serão analisadas as estatísticas descritivas dos valores dos bens, sua distribuição por meio de gráficos e os tipos de bens mais frequentes.

O objetivo é identificar padrões gerais, concentração de valores, assimetria e possíveis valores extremos (outliers).

### Estatísticas descritivas

A função `describe()` será utilizada para obter um resumo estatístico dos valores dos bens declarados, incluindo média, mediana, valores mínimo e máximo e quartis.

In [9]:
bens_df['VR_BEM_CANDIDATO'].describe()

count    1.654000e+03
mean     3.107977e+05
std      2.407173e+06
min      0.000000e+00
25%      1.000000e+04
50%      6.508050e+04
75%      2.256882e+05
max      9.200411e+07
Name: VR_BEM_CANDIDATO, dtype: float64

### Interpretação

A análise dos 1.654 bens declarados mostra uma distribuição bastante assimétrica. A média dos valores é de aproximadamente R$ 310.797,70, enquanto a mediana é de R$ 65.080,50. A diferença significativa entre esses valores indica que alguns bens possuem valores muito elevados e acabam aumentando a média. Isso é reforçado pelo valor máximo de aproximadamente R$ 92 milhões, enquanto 75% dos bens possuem valor de até aproximadamente R$ 225.688,20. Portanto, a maior parte dos bens está concentrada em valores relativamente menores, com alguns valores extremos influenciando fortemente a distribuição.

### Distribuição dos valores dos bens

O histograma permite observar como os valores dos bens estão distribuídos. A visualização será utilizada para verificar se os valores estão concentrados em determinadas faixas e se existem valores extremos.

In [10]:
fig = px.histogram(
    bens_df,
    x='VR_BEM_CANDIDATO',
    nbins=50,
    title='Distribuição do valor dos bens declarados',
    labels={
        'VR_BEM_CANDIDATO': 'Valor do bem (R$)'
    }
)

fig.show()

### Boxplot dos valores dos bens

O boxplot será utilizado para visualizar a dispersão dos valores e identificar possíveis outliers.

In [11]:
fig = px.box(
    bens_df,
    y='VR_BEM_CANDIDATO',
    title='Boxplot dos valores dos bens declarados',
    labels={
        'VR_BEM_CANDIDATO': 'Valor do bem (R$)'
    }
)

fig.show()

### Interpretação

A distribuição dos valores dos bens apresenta assimetria positiva, com grande concentração de bens em valores mais baixos e alguns bens com valores muito elevados. Esses valores extremos aparecem como outliers no boxplot e fazem com que a média seja superior à mediana.

### Tipos de bens mais comuns

Será analisada a frequência dos tipos de bens declarados pelos candidatos. O gráfico de barras apresenta os dez tipos mais frequentes.

In [12]:
tipos_bem = (
    bens_df['DS_TIPO_BEM_CANDIDATO']
    .value_counts()
    .head(10)
    .reset_index()
)

tipos_bem.columns = ['tipo_bem', 'quantidade']

fig = px.bar(
    tipos_bem,
    x='quantidade',
    y='tipo_bem',
    orientation='h',
    title='10 tipos de bens mais comuns',
    labels={
        'quantidade': 'Quantidade',
        'tipo_bem': 'Tipo de bem'
    }
)

fig.show()

## A4. Agregação por candidato, JOIN e geração do CSV

A base de bens possui uma linha para cada bem declarado. Portanto, um mesmo candidato pode aparecer várias vezes.

Nesta etapa, os bens serão agrupados por `SQ_CANDIDATO` para calcular:

- o patrimônio total, por meio da soma dos valores dos bens;
- a quantidade de bens declarados;
- o tipo de bem principal, definido como o tipo mais frequente entre os bens do candidato.

Após essa agregação, será realizado um JOIN com a base de candidatos.

In [13]:
# Agrupando os bens por candidato
patrimonio_df = bens_df.groupby('SQ_CANDIDATO').agg(
    PATRIMONIO_TOTAL=('VR_BEM_CANDIDATO', 'sum'),
    QTD_BENS=('VR_BEM_CANDIDATO', 'count')
).reset_index()

display(patrimonio_df.head())

,SQ_CANDIDATO,PATRIMONIO_TOTAL,QTD_BENS
0,40002531438,310000.00,3
1,40002531439,1873.49,3
2,40002531440,3708000.00,5
3,40002531441,444971.36,7
4,40002531442,648676.96,3


In [14]:
# Juntando os dados dos bens por candidato com o tipo principal de bem declarado
tipo_principal = (
    bens_df.groupby('SQ_CANDIDATO')['DS_TIPO_BEM_CANDIDATO']
    .agg(lambda x: x.mode().iloc[0] if not x.mode().empty else None)
    .reset_index()
)

tipo_principal.columns = [
    'SQ_CANDIDATO',
    'TIPO_BEM_PRINCIPAL'
]

patrimonio_df = patrimonio_df.merge(
    tipo_principal,
    on='SQ_CANDIDATO',
    how='left'
)

display(patrimonio_df.head())

,SQ_CANDIDATO,PATRIMONIO_TOTAL,QTD_BENS,TIPO_BEM_PRINCIPAL
0,40002531438,310000.00,3,Casa
1,40002531439,1873.49,3,Depósito bancário em conta corrente no País
2,40002531440,3708000.00,5,Quotas ou quinhões de capital
3,40002531441,444971.36,7,Consórcio não contemplado
4,40002531442,648676.96,3,Apartamento


### JOIN com os dados dos candidatos

Será utilizado um `left join`, tendo a base de candidatos como referência.

Essa escolha permite preservar todos os candidatos, inclusive aqueles que não possuem bens registrados na base de patrimônio. Nesses casos, o patrimônio poderá ser posteriormente representado como zero.

In [15]:
# Juntando os dados de candidatos com o patrimônio
candidatos_df = candidatos_df[colunas_candidatos]

df = candidatos_df.merge(
    patrimonio_df,
    on='SQ_CANDIDATO',
    how='left'
)

print(df.shape)
display(df.head())

(469, 9)


,SQ_CANDIDATO,DS_CARGO,SG_PARTIDO,DS_GENERO,DS_GRAU_INSTRUCAO,NM_URNA_CANDIDATO,PATRIMONIO_TOTAL,QTD_BENS,TIPO_BEM_PRINCIPAL
0,40002551283,SENADOR,REDE,MASCULINO,SUPERIOR COMPLETO,ISMAEL MUNDURUKU,300000.0,1.0,Casa
1,40002535270,SENADOR,PSTU,MASCULINO,ENSINO MÉDIO COMPLETO,EVANDRO DE OLIVEIRA,7000.0,1.0,"Veículo automotor terrestre: caminhão, automóv..."
2,40002551320,DEPUTADO ESTADUAL,REDE,MASCULINO,SUPERIOR COMPLETO,YURA MARUBO,160000.0,1.0,Casa
3,40002551315,DEPUTADO ESTADUAL,PSOL,MASCULINO,ENSINO FUNDAMENTAL COMPLETO,SENHOR LUIZ AGRICULTOR,NaN,NaN,NaN
4,40002542952,DEPUTADO ESTADUAL,PL,MASCULINO,ENSINO MÉDIO INCOMPLETO,RAMON PEDROSA,NaN,NaN,NaN


In [16]:
# Tratando quem não possui bens
df['PATRIMONIO_TOTAL'] = (
    df['PATRIMONIO_TOTAL'].fillna(0)
)

df['QTD_BENS'] = (
    df['QTD_BENS']
    .fillna(0)
    .astype(int)
)

df['TIPO_BEM_PRINCIPAL'] = (
    df['TIPO_BEM_PRINCIPAL']
    .fillna('Nenhum bem declarado')
)

display(df.head())


,SQ_CANDIDATO,DS_CARGO,SG_PARTIDO,DS_GENERO,DS_GRAU_INSTRUCAO,NM_URNA_CANDIDATO,PATRIMONIO_TOTAL,QTD_BENS,TIPO_BEM_PRINCIPAL
0,40002551283,SENADOR,REDE,MASCULINO,SUPERIOR COMPLETO,ISMAEL MUNDURUKU,300000.0,1,Casa
1,40002535270,SENADOR,PSTU,MASCULINO,ENSINO MÉDIO COMPLETO,EVANDRO DE OLIVEIRA,7000.0,1,"Veículo automotor terrestre: caminhão, automóv..."
2,40002551320,DEPUTADO ESTADUAL,REDE,MASCULINO,SUPERIOR COMPLETO,YURA MARUBO,160000.0,1,Casa
3,40002551315,DEPUTADO ESTADUAL,PSOL,MASCULINO,ENSINO FUNDAMENTAL COMPLETO,SENHOR LUIZ AGRICULTOR,0.0,0,Nenhum bem declarado
4,40002542952,DEPUTADO ESTADUAL,PL,MASCULINO,ENSINO MÉDIO INCOMPLETO,RAMON PEDROSA,0.0,0,Nenhum bem declarado


### Verificação do resultado final

Após o JOIN, será verificado se o resultado possui uma única linha por candidato. Essa verificação é importante para garantir que nenhum candidato tenha sido duplicado durante o processo de agregação e relacionamento das bases.

In [17]:
print("Linhas:", len(df))
print("Candidatos únicos:", df['SQ_CANDIDATO'].nunique())
print(
    "Duplicados:",
    df['SQ_CANDIDATO'].duplicated().sum()
)

Linhas: 469
Candidatos únicos: 469
Duplicados: 0


### Geração do CSV

O DataFrame final será salvo no arquivo `bens_am_por_candidato.csv`, que será utilizado posteriormente pelo aplicativo.

O arquivo conterá os principais atributos dos candidatos e as informações patrimoniais agregadas.

In [18]:
df.to_csv(
    'bens_am_por_candidato.csv',
    index=False,
    encoding='utf-8-sig'
)

### Maiores patrimônios

Por fim, os registros serão ordenados pelo patrimônio total em ordem decrescente para visualizar os candidatos com maiores valores declarados.

In [19]:
df_ordenado = df.sort_values(
    'PATRIMONIO_TOTAL',
    ascending=False
)

display(df_ordenado.head(10))

,SQ_CANDIDATO,DS_CARGO,SG_PARTIDO,DS_GENERO,DS_GRAU_INSTRUCAO,NM_URNA_CANDIDATO,PATRIMONIO_TOTAL,QTD_BENS,TIPO_BEM_PRINCIPAL
465,40002541626,GOVERNADOR,PL,FEMININO,SUPERIOR COMPLETO,PROFESSORA MARIA DO CARMO,1.184738e+08,27,Outras participações societárias
405,40002531447,SENADOR,MDB,MASCULINO,SUPERIOR COMPLETO,EDUARDO BRAGA,6.027856e+07,81,"Aplicação de renda fixa (CDB, RDB e outros)"
287,40002542688,1º SUPLENTE,PL,MASCULINO,SUPERIOR COMPLETO,ALESSANDRO TONIZA,2.344056e+07,44,Outras aplicações e Investimentos
166,40002532780,DEPUTADO ESTADUAL,PSD,MASCULINO,SUPERIOR COMPLETO,DR SANDOVAL CHEIRO DO POVO,2.324682e+07,17,"Veículo automotor terrestre: caminhão, automóv..."
91,40002541745,DEPUTADO ESTADUAL,UNIÃO,MASCULINO,SUPERIOR COMPLETO,ADJUTO AFONSO,1.054397e+07,27,"Aplicação de renda fixa (CDB, RDB e outros)"
293,40002542954,DEPUTADO ESTADUAL,PL,MASCULINO,ENSINO FUNDAMENTAL INCOMPLETO,MANOEL CUNHA,8.268302e+06,5,Prédio comercial
311,40002532744,DEPUTADO ESTADUAL,MDB,MASCULINO,SUPERIOR COMPLETO,SERGIO NUNES,8.154000e+06,2,Terreno
310,40002532749,DEPUTADO ESTADUAL,MDB,MASCULINO,SUPERIOR INCOMPLETO,SIMÃO PEIXOTO,7.205806e+06,11,Casa
62,40002531448,1º SUPLENTE,MDB,FEMININO,SUPERIOR COMPLETO,SANDRA BRAGA,6.435907e+06,17,"Aplicação de renda fixa (CDB, RDB e outros)"
215,40002536433,DEPUTADO FEDERAL,AVANTE,MASCULINO,SUPERIOR COMPLETO,CORONEL MENEZES,5.131698e+06,10,"Aplicação de renda fixa (CDB, RDB e outros)"


## A5. EDA robusta após o JOIN

Com o patrimônio agregado e os atributos dos candidatos reunidos em uma única base, torna-se possível comparar o patrimônio entre diferentes grupos.

Serão realizadas comparações entre patrimônio e cargo, patrimônio e gênero e, adicionalmente, patrimônio e partido.

### Patrimônio mediano por cargo

A mediana será utilizada para comparar os cargos porque os valores patrimoniais apresentam grande dispersão e possíveis outliers. Dessa forma, a mediana representa melhor o patrimônio típico de cada grupo.

In [20]:
patrimonio_cargo = (
    df
    .groupby('DS_CARGO')['PATRIMONIO_TOTAL']
    .median()
    .sort_values(ascending=False)
    .reset_index()
)

fig = px.bar(
    patrimonio_cargo,
    x='DS_CARGO',
    y='PATRIMONIO_TOTAL',
    title='Patrimônio mediano por cargo',
    labels={
        'DS_CARGO': 'Cargo',
        'PATRIMONIO_TOTAL': 'Patrimônio mediano (R$)'
    }
)

fig.update_layout(xaxis_tickangle=-45)

fig.show()

### Patrimônio por partido

O boxplot será utilizado para comparar a distribuição do patrimônio entre os grupos de partidos, permitindo observar diferenças nas medianas e na dispersão dos valores.

In [21]:
patrimonio_partido = (
    df
    .groupby('SG_PARTIDO')['PATRIMONIO_TOTAL']
    .median()
    .sort_values(ascending=False)
    .reset_index()
)

fig = px.bar(
    patrimonio_partido,
    x='SG_PARTIDO',
    y='PATRIMONIO_TOTAL',
    title='Patrimônio mediano por partido',
    labels={
        'SG_PARTIDO': 'Partido',
        'PATRIMONIO_TOTAL': 'Patrimônio mediano (R$)'
    }
)
fig.update_layout(xaxis_tickangle=-45)
fig.show()

### Patrimônio por gênero

O boxplot será utilizado para comparar a distribuição do patrimônio entre os grupos de gênero, permitindo observar diferenças nas medianas e na dispersão dos valores.

In [22]:
fig = px.box(
    df,
    x='DS_GENERO',
    y='PATRIMONIO_TOTAL',
    title='Distribuição do patrimônio por gênero',
    labels={
        'DS_GENERO': 'Gênero',
        'PATRIMONIO_TOTAL': 'Patrimônio total (R$)'
    }
)

fig.show()

In [23]:
df.groupby(
    'DS_GENERO'
)['PATRIMONIO_TOTAL'].median().sort_values(
    ascending=False
)

DS_GENERO
MASCULINO    200000.0
FEMININO       5000.0
Name: PATRIMONIO_TOTAL, dtype: float64

### Interpretação

#### Patrimônio por cargo:

O patrimônio mediano é claramente maior nos cargos majoritários: Governador lidera (~R$ 1,65 milhão), seguido por Vice-Governador e Senador. Deputado Federal e as suplências ficam entre as menores medianas, evidenciando perfis patrimoniais distintos entre candidaturas majoritárias e proporcionais.

#### Patrimônio por partido:

O PP concentra a maior mediana patrimonial (acima de R$ 1,6 milhão), destacando-se de partidos como MDB, PSDB, União e Avante. Legendas menores, como Novo, PSTU e PCdoB, têm mediana próxima de zero, refletindo diferenças na base social das candidaturas e no volume de candidatos lançados por partido.

#### Patrimônio por gênero:

O patrimônio mediano entre candidatos masculinos (R$ 200.000) é 40 vezes maior que entre candidatas femininas (R$ 5.000). O boxplot também mostra outliers expressivos em ambos os grupos, indicando que alguns candidatos concentram patrimônios muito acima da distribuição típica de seu gênero.

# A6. Narrativa — Storytelling

## Contexto

A análise reúne os candidatos do Amazonas nas Eleições 2026 e os bens que declararam ao Tribunal Superior Eleitoral (TSE). A partir da agregação dos bens por candidato (soma dos valores, quantidade de bens e tipo de bem principal) e do JOIN com os atributos de cada candidatura — cargo, partido e gênero —, foi possível ir além da distribuição geral de bens e investigar como o patrimônio se comporta entre diferentes perfis de candidatos.

## Conflito / Insight

A análise univariada dos 1.654 bens declarados já revelou uma distribuição fortemente assimétrica: a média (R$ 310.797,70) fica quase cinco vezes acima da mediana (R$ 65.080,50), e o valor máximo declarado chega a cerca de R$ 92 milhões, contra um terceiro quartil de apenas R$ 225.688,20. Ou seja, a maioria dos bens tem valor modesto, mas um pequeno número de bens de altíssimo valor puxa a média para cima — padrão confirmado pelos outliers visíveis no boxplot.

Esse mesmo padrão de concentração se repete, e se aprofunda, quando o patrimônio é agregado por candidato e cruzado com cargo, partido e gênero:

- **Por cargo:** o patrimônio mediano varia fortemente conforme a posição disputada. Candidatos a **Governador** têm a maior mediana (cerca de R$ 1,65 milhão), seguidos por Vice-Governador (~R$ 0,45 milhão) e Senador (~R$ 0,30 milhão), enquanto Deputado Federal aparece entre os menores (~R$ 0,04 milhão) — uma diferença de mais de 40 vezes entre os extremos.
- **Por partido:** o PP concentra a maior mediana patrimonial (acima de R$ 1,6 milhão), destacando-se de partidos como MDB, PSDB, União e Avante (todos na faixa de R$ 0,5–0,7 milhão), enquanto legendas como Novo, PSTU, PCdoB, Rede e DC têm mediana próxima de zero — sugerindo que o partido também é um proxy relevante para o perfil socioeconômico das candidaturas.
- **Por gênero:** a disparidade é a mais evidente da análise. A mediana entre candidatos masculinos (R$ 200.000) é 40 vezes maior que a das candidatas femininas (R$ 5.000). O boxplot também mostra que os maiores outliers aparecem nos dois grupos — como o de Professora Maria do Carmo (candidata a Governadora, ~R$ 118,5 milhões) e Eduardo Braga (candidato a Senador, ~R$ 60,3 milhões) —, o que reforça que a desigualdade de gênero está na *distribuição típica*, e não apenas nos casos extremos.

## Conclusão

Os dados mostram que o patrimônio declarado pelos candidatos do Amazonas em 2026 é marcado por forte concentração e desigualdade, tanto na distribuição geral dos bens quanto nas comparações entre cargo, partido e gênero. O JOIN entre as bases de candidatos e de bens foi essencial para essa análise: sem ele, seria possível descrever apenas a distribuição bruta dos valores dos bens, mas não relacioná-la aos perfis das candidaturas. Ao unir as duas bases, foi possível identificar que cargos mais altos na hierarquia política (Governador, Vice-Governador, Senador) concentram os maiores patrimônios medianos, que alguns partidos (como o PP) lançam candidaturas de perfil patrimonial bem mais alto que a maioria das legendas, e que existe uma disparidade patrimonial expressiva entre candidatos e candidatas — achado que, por si só, já sugere uma linha de investigação futura sobre barreiras de acesso e financiamento de campanhas por gênero.